## Training Task: Sequence Reversal

Reverse a source sequence of length $N$ drawn from vocabulary $V$.

**Why this tests the decoder specifically:**

* **Cross-attention is required** — the decoder must look at memory (the embedded source) to know
  what the reversed tokens are. A model that ignores cross-attention cannot solve this.
* **Causal self-attention is tested** — teacher forcing with an upper-triangular $-\infty$ mask
  prevents the decoder from peeking at future targets during training.
* **Autoregressive decoding** — after training we verify the decoder generates correctly one token
  at a time (no teacher forcing), which tests the full inference loop.

Architecture (decoder-only, encoder replaced by a learned embedding):

$$\text{memory} = \text{Embed}_{\text{src}}(\mathbf{s}) \qquad
  \hat{\mathbf{t}} = \text{head}(\text{Decoder}(\text{Embed}_{\text{tgt}}(\mathbf{t}_{\text{in}}),\;\text{memory}))$$

In [ ]:
import math

import matplotlib.pyplot as plt
import torch
import torch.nn as nn

import models.deep_learning.architectures as mynn
from helpers import get_device

In [ ]:
device = get_device()
dtype = torch.float32

## Hyper-parameters

In [ ]:
V = 8  # token vocabulary  {0 … 7}
SEQ_LEN = 5  # source / target length
D = 32  # d_model (larger than the unit-test above to give capacity)
NHEAD = 4
NLAYERS = 2
FF = 128
BOS = V  # BOS index (= V, one beyond the regular vocab)

## Data 
 * src:     (B, SEQ_LEN) – random integer sequences
 * tgt_in:  (B, SEQ_LEN) – BOS + reversed src (drop last token)  ← decoder input
 * tgt_out: (B, SEQ_LEN) – reversed src                          ← prediction target

In [ ]:
def make_batch(batch_size: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    src = torch.randint(0, V, (batch_size, SEQ_LEN), device=device)
    rev = src.flip(1)
    bos_col = torch.full((batch_size, 1), BOS, device=device)
    tgt_in = torch.cat([bos_col, rev[:, :-1]], dim=1)  # BOS + first SEQ_LEN-1 of rev
    tgt_out = rev  # full reversed src
    return src, tgt_in, tgt_out

##  Model
Positional embeddings are required: without them the cross-attention is
permutation-equivariant over memory and cannot learn "attend to the last
source position first."  Token + position embeddings are summed, following
the standard transformer word-encoding pattern.

In [ ]:
torch.manual_seed(0)
src_tok_emb = nn.Embedding(V, D, device=device)  # source token embeddings
src_pos_emb = nn.Embedding(SEQ_LEN, D, device=device)  # source positional embeddings
tgt_tok_emb = nn.Embedding(V + 1, D, device=device)  # +1 for BOS
tgt_pos_emb = nn.Embedding(SEQ_LEN, D, device=device)  # target positional embeddings

src_pos = torch.arange(SEQ_LEN, device=device).unsqueeze(0)  # (1, SEQ_LEN)

dec_layer = mynn.TransformerDecoderLayer(
    D,
    NHEAD,
    dim_feedforward=FF,
    dropout=0.0,
    activation_cls=nn.GELU,
    norm_first=True,
    device=device,
    dtype=dtype,
)

nn_dec_layer = nn.TransformerDecoderLayer(
    D,
    NHEAD,
    dim_feedforward=FF,
    dropout=0.0,
    activation="gelu",
    batch_first=True,
    norm_first=True,
    device=device,
    dtype=dtype,
)

rev_decoder = mynn.TransformerDecoder(
    dec_layer, NLAYERS, norm=nn.LayerNorm(D, device=device, dtype=dtype)
)
rev_head = nn.Linear(D, V, device=device, dtype=dtype)

### Compare with PyTorch's built-in decoder, to validate our implementation

In [ ]:
nn_rev_decoder = nn.TransformerDecoder(
    nn_dec_layer, NLAYERS, norm=nn.LayerNorm(D, device=device, dtype=dtype)
)
rev_decoder.load_weights_from_torch_decoder(nn_rev_decoder)

In [ ]:
all_params = (
    list(src_tok_emb.parameters())
    + list(src_pos_emb.parameters())
    + list(tgt_tok_emb.parameters())
    + list(tgt_pos_emb.parameters())
    + list(rev_decoder.parameters())
    + list(rev_head.parameters())
)
optimizer = torch.optim.Adam(all_params, lr=3e-4)
criterion = nn.CrossEntropyLoss()

# Causal mask: prevents each decoder position from attending to future positions.
tgt_causal_mask = torch.triu(
    torch.full((SEQ_LEN, SEQ_LEN), float("-inf"), device=device), diagonal=1
)

print(f"Parameters: {sum(p.numel() for p in all_params):,}")
print(f"Random-baseline cross-entropy ≈ log({V}) = {math.log(V):.3f}")

## Trianing

In [ ]:
EPOCHS = 150
BATCH_SIZE = 256

train_losses = []
for epoch in range(1, EPOCHS + 1):
    src_tok_emb.train()
    src_pos_emb.train()
    tgt_tok_emb.train()
    tgt_pos_emb.train()
    rev_decoder.train()
    rev_head.train()

    src, tgt_in, tgt_out = make_batch(BATCH_SIZE)

    memory = src_tok_emb(src) + src_pos_emb(src_pos)  # (B, SEQ_LEN, D)
    tgt_e = tgt_tok_emb(tgt_in) + tgt_pos_emb(src_pos)  # (B, SEQ_LEN, D)

    out = rev_decoder(tgt_e, memory, tgt_mask=tgt_causal_mask)  # (B, SEQ_LEN, D)
    logits = rev_head(out)  # (B, SEQ_LEN, V)
    loss = criterion(logits.reshape(-1, V), tgt_out.reshape(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    train_losses.append(loss.item())
    if epoch % 50 == 0:
        print(f"epoch {epoch:3d}  loss={loss.item():.4f}")

## Training loss

In [ ]:
plt.figure(figsize=(7, 3))
plt.plot(train_losses)
plt.axhline(
    y=math.log(V),
    color="gray",
    linestyle="--",
    linewidth=0.8,
    label=f"random baseline log({V})",
)
plt.xlabel("epoch")
plt.ylabel("cross-entropy loss")
plt.title(f"Sequence reversal  (V={V}, N={SEQ_LEN})")
plt.legend()
plt.tight_layout()
plt.show()

## Evaluaiton

Evaluate on fresh examples

In [ ]:
N_TEST = 20


@torch.no_grad()
def greedy_decode(src_seq: torch.Tensor) -> list[int]:
    """Autoregressive decoding — no teacher forcing, one token at a time."""
    src_tok_emb.eval()
    src_pos_emb.eval()
    tgt_tok_emb.eval()
    tgt_pos_emb.eval()
    rev_decoder.eval()
    rev_head.eval()

    mem = src_tok_emb(src_seq.unsqueeze(0)) + src_pos_emb(src_pos)  # (1, SEQ_LEN, D)
    tgt = torch.tensor([[BOS]], device=device)

    for step in range(SEQ_LEN):
        step_pos = torch.arange(step + 1, device=device).unsqueeze(0)
        tgt_e = tgt_tok_emb(tgt) + tgt_pos_emb(step_pos)  # (1, step+1, D)
        mask = torch.triu(
            torch.full((step + 1, step + 1), float("-inf"), device=device), diagonal=1
        )
        out = rev_decoder(tgt_e, mem, tgt_mask=mask)
        next_tok = rev_head(out[:, -1]).argmax(-1, keepdim=True)
        tgt = torch.cat([tgt, next_tok], dim=1)

    return tgt[0, 1:].tolist()  # strip BOS


correct = 0
print("src                 expected            predicted        ok")
print("─" * 60)
for _ in range(N_TEST):
    src_seq = torch.randint(0, V, (SEQ_LEN,), device=device)
    expected = src_seq.flip(0).tolist()
    pred = greedy_decode(src_seq)
    ok = pred == expected
    correct += int(ok)
    print(f"{src_seq.tolist()}  →  {expected}  →  {pred}  {'✓' if ok else '✗'}")

print(f"\nAccuracy: {correct}/{N_TEST} = {correct / N_TEST:.0%}")